<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F%206.7.%20%D0%A1%D1%83%D0%BF%D0%B5%D1%80%D0%B2%D0%B0%D0%B9%D0%B7%D0%B5%D1%80%20%D0%B8%20%D1%81%D0%BB%D0%BE%D0%B6%D0%BD%D0%B0%D1%8F%20%D0%BE%D1%80%D0%BA%D0%B5%D1%81%D1%82%D1%80%D0%B0%D1%86%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.7. Супервайзер и сложная оркестрация

## Введение: от команды равных к иерархии с управляющим

В Лекции 6.6 мы построили multi‑agent систему, где исследователь, критик и писатель работали в цикле, обмениваясь сообщениями и уточняя результаты. Это была **команда равных** – каждый агент выполнял свою роль, но никто не управлял процессом. Мы жёстко задали порядок: исследователь → критик → (возврат) → писатель. Это работало, но было негибко.

Представьте, что вы – руководитель проекта. Вы не просто даёте задание и ждёте результат. Вы оцениваете промежуточные итоги, перераспределяете задачи, меняете приоритеты, подключаете новых экспертов, если это нужно. Именно так работает **супервайзер** – управляющий агент, который координирует работу других агентов, ставит им задачи, собирает результаты и принимает решения о следующих шагах.

В этой лекции мы добавим в нашу multi‑agent систему **супервайзера**. Он будет не просто передавать управление по кругу, а **динамически** решать, какого агента вызвать, когда завершить работу и какие дополнительные действия предпринять. Это сделает систему по‑настоящему гибкой, способной адаптироваться к неожиданным результатам и сложным сценариям.

Мы реализуем супервайзера как отдельного LLM-агента, который видит всю историю диалога, знает, какие агенты доступны, и использует их как «инструменты» для достижения цели. В финале мы увидим, как супервайзер может перепланировать работу на лету, если первоначальный план не сработал.

---

## Тема 1. Зачем нужен супервайзер

### 1.1. Отличие от простой цепочки

В предыдущих лекциях мы использовали **жёсткие цепочки** и **условные переходы**. Например, в Лекции 6.6 мы зафиксировали порядок: исследователь → критик → (если нужно, снова исследователь) → писатель. Это работает для многих задач, но имеет ограничения:

- **Нет выбора** – мы не можем в зависимости от промежуточного результата вызвать другого агента (например, аналитика вместо исследователя).
- **Нет параллельности** – мы не можем запустить двух агентов одновременно и объединить их результаты.
- **Нет стратегического планирования** – мы не можем сказать: «Сначала собери факты, потом проверь их, а если найдешь противоречия – поищи дополнительную информацию в другом источнике».

Супервайзер решает эти проблемы. Он видит всю картину и может перепланировать. Вместо фиксированного графа он принимает решения на каждом шаге:

1. **Какой агент нужен сейчас?** – Исследователь, критик, писатель или кто‑то ещё.
2. **Какую задачу ему дать?** – Не просто «собери факты», а «найди информацию о компании X за 2023 год».
3. **Когда остановиться?** – Если цель достигнута, завершить работу.

Это похоже на то, как менеджер проекта распределяет задачи между разработчиками, тестировщиками и аналитиками, корректируя план по ходу дела.

### 1.2. Примеры использования супервайзера

Супервайзер особенно полезен в сценариях, где:

- **Автоматизация бизнес-процессов** – обработка заявок, генерация отчётов, взаимодействие с клиентами.
- **Исследовательские проекты** – сбор данных из разных источников, проверка гипотез, синтез знаний.
- **Написание кода** – один агент пишет код, другой тестирует, третий ревьюит, а супервайзер решает, когда переходить к следующему этапу.
- **Клиентская поддержка** – супервайзер определяет, нужна ли техническая информация, юридическая консультация или просто разговор с оператором.

В каждом из этих случаев супервайзер не просто передаёт управление, а активно анализирует прогресс и адаптирует стратегию.

### 1.3. Как супервайзер принимает решения

Супервайзер принимает решения на основе **текущего состояния** системы. Состояние включает:

- **Историю сообщений** – что говорили агенты и пользователь.
- **Промежуточные результаты** – факты, гипотезы, ошибки.
- **Метрики прогресса** – сколько информации уже собрано, насколько она полна.

На каждом шаге супервайзер анализирует состояние и выбирает одно из действий:

- **Вызвать агента** – передать задачу конкретному специалисту.
- **Завершить работу** – если цель достигнута.
- **Запросить уточнение у пользователя** – если не хватает данных или есть неоднозначность.

Для этого супервайзер использует языковую модель, которая получает описание текущего состояния и список доступных агентов (как инструментов) и возвращает решение. Это похоже на то, как в Лекции 6.4 мы привязывали инструменты к агенту – теперь агенты сами становятся «инструментами» для супервайзера.

### 1.4. Реализация супервайзера в LangGraph

В LangGraph супервайзер – это обычный узел графа, но с особым поведением. Он:

1. **Принимает состояние** (историю, результаты).
2. **Вызывает LLM** с промптом, который описывает доступных агентов и цель.
3. **Возвращает решение** – имя следующего агента или команду завершить.

На практике супервайзер может быть реализован как:

```python
def supervisor_node(state: dict) -> dict:
    # Получаем историю и текущий прогресс
    messages = state["messages"]
    progress = state.get("progress", "")
    
    # Формируем промпт для супервайзера
    prompt = f"""
    Ты – супервайзер. Управляй командой агентов:
    - researcher: ищет факты
    - critic: проверяет достоверность
    - writer: пишет ответ
    
    История: {messages}
    Прогресс: {progress}
    
    Выбери следующего агента или заверши работу.
    """
    response = llm.invoke(prompt)
    decision = response.content.strip()
    
    # Возвращаем решение
    return {"next_agent": decision}
```

Затем мы добавляем условное ребро, которое направляет управление в зависимости от решения супервайзера. В следующих темах мы детально реализуем этот механизм и покажем, как он делает систему по‑настоящему гибкой.

---

В следующей части мы перейдём к проектированию графа с супервайзером и создадим полноценную оркестрированную multi‑agent систему.

# Лекция 6.7. Супервайзер и сложная оркестрация

## Тема 3. Реализация супервайзера как LLM (скрипт `supervisor_graph_llm.py`)

В предыдущей теме мы спроектировали граф с супервайзером и описали состояние. Теперь мы переходим к реализации супервайзера как **LLM-агента**, который принимает решения на основе анализа текущего состояния и истории. Именно такой подход мы использовали в вашем рабочем скрипте `supervisor_graph_llm.py`, который уже успешно продемонстрировал свою работу.

В этой теме мы детально разберём код, объясним, как супервайзер использует инструменты для вызова агентов, и покажем, как это работает на практике.

---

### 3.1. Почему LLM, а не правила?

Эвристический супервайзер (например, с простыми условиями «если нет фактов → researcher») работает только в самых простых сценариях. В реальных задачах возникает множество нюансов:

- **Неоднозначность** – запрос может требовать и фактов, и анализа, и творчества.
- **Динамика** – промежуточные результаты могут менять план.
- **Контекст** – история диалога может влиять на следующее действие.

LLM-супервайзер преодолевает эти ограничения. Он:

- **Понимает сложные запросы** – например, «Сравни RAG и обычный ChatGPT» может потребовать нескольких шагов исследования, проверки и синтеза.
- **Адаптируется к нестандартным ситуациям** – если критик нашёл противоречия, супервайзер может вернуть исследователя для уточнения.
- **Учитывает историю** – он не повторяет уже выполненные шаги и видит прогресс.

Таким образом, LLM-супервайзер делает систему по‑настоящему гибкой и интеллектуальной, что мы и наблюдаем в вашем работающем коде.

---

### 3.2. Представление агентов как «инструментов»

В LangChain агенты могут вызывать инструменты через `bind_tools()`. В нашем случае супервайзер не выполняет агентов напрямую, а только выбирает, какого агента вызвать. Для этого мы определяем функции-заглушки для каждого агента, которые служат описанием их возможностей. Эти функции привязываются к LLM супервайзера через `bind_tools()`, и модель возвращает `tool_calls` с именем нужного агента.

**В вашем коде это выглядит так:**

```python
from langchain.tools import tool

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки фактов."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа."""
    return "Писатель напишет ответ."

supervisor_tools = [call_researcher, call_critic, call_writer]
llm_with_tools = llm.bind_tools(supervisor_tools)
```

**Что здесь происходит:**
- Каждая функция снабжена docstring, который описывает её назначение.
- LLM видит эти описания и может решить, какой «инструмент» (агент) нужен.
- `bind_tools()` подготавливает модель к возврату структурированного ответа с `tool_calls`.

Когда супервайзер вызывает LLM, модель возвращает либо `tool_calls` с именем агента, либо текстовый ответ, если она решила завершить работу. Мы парсим `tool_calls` и извлекаем имя агента.

---

### 3.3. Системный промпт для супервайзера

Системный промпт задаёт правила и контекст для LLM. В вашем коде он выглядит так:

```python
SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к трём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики

Процесс должен быть логичным:
1. Если нет фактов — вызови researcher.
2. Если факты есть, но нет критики — вызови critic.
3. Если есть и факты, и критика, но нет ответа — вызови writer.
4. Если ответ уже есть — заверши работу.

Твоя задача — анализировать текущее состояние и выбирать правильный следующий шаг.
"""
```

Этот промпт даёт модели чёткое понимание её роли и правил принятия решений. Он также описывает доступные инструменты, что помогает модели выбирать правильный `tool_call`.

---

### 3.4. Полный код супервайзера с LLM (ваш рабочий файл)

Ниже представлен полный код `supervisor_graph_llm.py`, который вы успешно запустили. Он включает все компоненты: состояние, инструменты, узлы, маршрутизацию и тестовый запуск.

```python
"""
supervisor_graph_llm.py - Граф с супервайзером на основе LLM (исправленная версия)
Лекция 6.7, Тема 3
"""

from typing import TypedDict, List, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool

# ============================================================================
# 1. ОПРЕДЕЛЕНИЕ СОСТОЯНИЯ
# ============================================================================

class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    next_agent: Literal["researcher", "critic", "writer", "finish"]
    research_result: Optional[str]
    critic_feedback: Optional[str]
    final_answer: Optional[str]
    iteration: int
    history_summary: str
    error: Optional[str]

# ============================================================================
# 2. ОПРЕДЕЛЕНИЕ «ИНСТРУМЕНТОВ» ДЛЯ СУПЕРВАЙЗЕРА
# ============================================================================

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов. Используй, если нужна новая информация."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки достоверности и полноты фактов. Вызывай после получения research_result."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа. Вызывай после того, как есть и research_result, и critic_feedback."""
    return "Писатель напишет ответ."

supervisor_tools = [call_researcher, call_critic, call_writer]

# ============================================================================
# 3. ИНИЦИАЛИЗАЦИЯ LLM И ПРОМПТА
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)
llm_with_tools = llm.bind_tools(supervisor_tools)

SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к трём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики

Процесс должен быть логичным:
1. Если нет фактов — вызови researcher.
2. Если факты есть, но нет критики — вызови critic.
3. Если есть и факты, и критика, но нет ответа — вызови writer.
4. Если ответ уже есть — заверши работу (не вызывай инструменты).

Твоя задача — анализировать текущее состояние и выбирать правильный следующий шаг.
Ты можешь вызвать только один инструмент за раз (или завершить).
"""

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def supervisor_node(state: SupervisorState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🎯 Супервайзер (итерация {iteration})")

    # Защита от бесконечного цикла (hard limit)
    if iteration > 5:
        print("⚠️  Превышен лимит итераций. Принудительное завершение.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено принудительно"
        }

    # 🛠 ИСПРАВЛЕНИЕ 1: Немедленный выход, если ответ уже готов
    final_ans = state.get("final_answer")
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено (ответ уже есть)"
        }

    # Формируем контекст для LLM
    question = state.get("question", "")
    research = state.get("research_result")
    critic_fb = state.get("critic_feedback")

    # Строим историю сообщений (для контекста)
    history_text = "\n".join([f"{msg.type}: {msg.content[:200]}" for msg in state.get("messages", [])])

    user_content = f"""
Вопрос пользователя: {question}

Текущий прогресс:
- Исследование: {research if research else "ещё нет"}
- Критика: {critic_fb if critic_fb else "ещё нет"}
- Финальный ответ: {final_ans if final_ans else "ещё нет"}

История сообщений:
{history_text}

Выбери следующего агента (researcher, critic, writer) или заверши работу (finish).
Вызови соответствующий инструмент.
"""

    messages = [
        SystemMessage(content=SUPERVISOR_PROMPT),
        HumanMessage(content=user_content)
    ]

    # Вызов LLM с инструментами
    response = llm_with_tools.invoke(messages)
    print(f"💬 Ответ LLM: {response.content[:100]}...")

    # Извлечение tool_calls
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_name = response.tool_calls[0]["name"]
        # Преобразуем имя инструмента к нашему формату
        if "researcher" in tool_name:
            proposed = "researcher"
        elif "critic" in tool_name:
            proposed = "critic"
        elif "writer" in tool_name:
            proposed = "writer"
        else:
            proposed = "finish"
    else:
        # Если модель не вызвала инструмент, пытаемся извлечь из текста
        text = response.content.lower()
        if "researcher" in text:
            proposed = "researcher"
        elif "critic" in text:
            proposed = "critic"
        elif "writer" in text:
            proposed = "writer"
        else:
            proposed = "finish"

    # Валидация предложения (корректируем, если логика нарушена)
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу (валидация).")
        proposed = "finish"
    elif proposed == "critic" and research is None:
        print("⚠️  LLM вызвала критика без исследования. Перенаправляем на исследователя.")
        proposed = "researcher"
    elif proposed == "writer" and (research is None or critic_fb is None):
        print("⚠️  LLM вызвала писателя без полных данных (исследование или критика отсутствуют). Перенаправляем на критика.")
        if research is None:
            proposed = "researcher"
        else:
            proposed = "critic"
    elif proposed == "researcher" and research is not None and critic_fb is not None:
        print("⚠️  LLM предлагает исследовать, хотя данные уже есть. Перенаправляем на писателя.")
        proposed = "writer"

    # Если после валидации получился "finish", но не все данные собраны — корректируем
    if proposed == "finish" and final_ans is None:
        if research is None:
            proposed = "researcher"
        elif critic_fb is None:
            proposed = "critic"
        else:
            proposed = "writer"

    print(f"🔀 Супервайзер выбрал: {proposed.upper()}")
    return {
        "next_agent": proposed,
        "iteration": iteration,
        "history_summary": f"Итерация {iteration}: выбран {proposed}"
    }


def researcher_node(state: SupervisorState) -> dict:
    print("🔍 Исследователь: собираю факты...")
    question = state.get("question", "")
    # Имитация поиска (можно заменить на реальный RAG)
    if "RAG" in question.upper():
        facts = "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста на основе найденных документов. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in question.upper():
        facts = "Большие языковые модели (LLM) обучаются на больших объёмах текстов и способны генерировать связные ответы, выполнять перевод, реферирование и многие другие задачи."
    else:
        facts = "Информация по запросу не найдена. Возможно, требуется уточнить вопрос."
    msg = AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{facts}")
    return {
        "research_result": facts,
        "messages": [msg]
    }


def critic_node(state: SupervisorState) -> dict:
    print("📊 Критик: проверяю факты...")
    research = state.get("research_result", "")
    if "не найдена" in research.lower():
        feedback = "Факты неполные. Рекомендую провести дополнительный поиск."
    elif len(research) < 30:
        feedback = "Факты слишком краткие. Желательно добавить больше деталей."
    else:
        feedback = "Факты достаточны и достоверны."
    msg = AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")
    return {
        "critic_feedback": feedback,
        "messages": [msg]
    }


def writer_node(state: SupervisorState) -> dict:
    print("✍️  Писатель: пишу финальный ответ...")
    research = state.get("research_result", "")
    feedback = state.get("critic_feedback")
    if feedback is None:
        feedback = "Критика не проводилась."
    else:
        feedback = feedback.lower()

    if "неполные" in feedback or "краткие" in feedback:
        answer = f"На основе найденных фактов:\n{research}\n\nПримечание: {feedback} Рекомендуется уточнить информацию."
    else:
        answer = f"На основе проверенных фактов:\n{research}"
    msg = AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{answer}")
    return {
        "final_answer": answer,
        "messages": [msg]
    }

# ============================================================================
# 5. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_supervisor(state: SupervisorState) -> Literal["researcher", "critic", "writer", "finish"]:
    next_agent = state.get("next_agent", "finish")
    print(f"🔀 Маршрутизация: {next_agent.upper()}")
    return next_agent

# ============================================================================
# 6. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "researcher": "researcher",
        "critic": "critic",
        "writer": "writer",
        "finish": END
    }
)
builder.add_edge("researcher", "supervisor")
builder.add_edge("critic", "supervisor")
builder.add_edge("writer", "supervisor")

graph = builder.compile()

# ============================================================================
# 7. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    initial_state = {
        "messages": [HumanMessage(content="Что такое RAG?")],
        "question": "Что такое RAG?",
        "next_agent": "researcher",
        "research_result": None,
        "critic_feedback": None,
        "final_answer": None,
        "iteration": 0,
        "history_summary": "Начало работы",
        "error": None
    }
    config = {"recursion_limit": 10}
    print("=" * 60)
    print("🚀 ЗАПУСК ГРАФА С LLM-СУПЕРВАЙЗЕРОМ (LLM главный)")
    print("=" * 60)
    try:
        result = graph.invoke(initial_state, config=config)
    except Exception as e:
        print(f"❌ Ошибка выполнения: {e}")
        result = None
    if result:
        print("\n" + "=" * 60)
        print("✅ ИТОГОВЫЙ РЕЗУЛЬТАТ")
        print("=" * 60)
        print(f"Финальный ответ: {result.get('final_answer', 'Не сгенерирован')}")
        print(f"Количество шагов: {len(result['messages'])}")
        print("\n📋 Полная история сообщений:")
        for i, msg in enumerate(result["messages"], 1):
            print(f"{i}. {msg.content[:100]}...")
```

---

### 3.5. Как это работает

1. **Супервайзер** (узел `supervisor_node`) получает состояние, включая вопрос, результаты агентов и историю.
2. Он формирует промпт с текущим прогрессом и вызывает LLM с привязанными инструментами.
3. LLM возвращает `tool_calls`, указывая, какого агента вызвать, или текстовый ответ, если решает завершить.
4. Мы извлекаем имя агента из `tool_calls` и сохраняем в `next_agent`.
5. **Валидация** корректирует ошибочные решения (например, если LLM вызывает критика без исследования).
6. **Маршрутизация** направляет управление соответствующему агенту.
7. После выполнения агента управление возвращается к супервайзеру.
8. Цикл повторяется, пока супервайзер не решит завершить работу (или не сработает защита от цикла).

**Результат работы вашего скрипта:**

```
🚀 ЗАПУСК ГРАФА С LLM-СУПЕРВАЙЗЕРОМ (LLM главный)
============================================================

🎯 Супервайзер (итерация 1)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: RESEARCHER
🔀 Маршрутизация: RESEARCHER
🔍 Исследователь: собираю факты...

🎯 Супервайзер (итерация 2)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: CRITIC
🔀 Маршрутизация: CRITIC
📊 Критик: проверяю факты...

🎯 Супервайзер (итерация 3)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: WRITER
🔀 Маршрутизация: WRITER
✍️  Писатель: пишу финальный ответ...

🎯 Супервайзер (итерация 4)
✅ Финальный ответ уже сгенерирован. Завершаем работу.
🔀 Маршрутизация: FINISH

============================================================
✅ ИТОГОВЫЙ РЕЗУЛЬТАТ
============================================================
Финальный ответ: На основе проверенных фактов:
RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста на основе найденных документов. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM).
```

---

### 3.6. Преимущества LLM-супервайзера

| Аспект | Эвристический супервайзер | LLM-супервайзер |
|--------|---------------------------|-----------------|
| **Гибкость** | Ограничена заданными правилами | Адаптируется к контексту |
| **Понимание** | Не понимает нюансов | Учитывает смысл и историю |
| **Масштабируемость** | Трудно добавить новые сценарии | Легко добавить новых агентов через инструменты |
| **Качество решений** | Предсказуемо, но не всегда оптимально | Может принимать нестандартные решения |

Ваш код демонстрирует, как LLM-супервайзер успешно управляет процессом, последовательно вызывая нужных агентов и завершая работу, когда ответ готов.

---

## Краткий итог Тема 3

- **LLM-супервайзер** заменяет эвристические правила, делая управление более интеллектуальным.
- **Агенты представлены как инструменты** через `bind_tools()`, что позволяет супервайзеру «вызывать» их.
- **Валидация решений** предотвращает логические ошибки (например, вызов критика без исследования).
- **Граф** сохраняет структуру: супервайзер → агент → супервайзер → … → завершение.
- **Защита от бесконечного цикла** реализована через `iteration` и `max_iterations`.

Теперь ваша система стала по‑настоящему адаптивной. В следующей теме мы рассмотрим, как супервайзер может управлять более сложными сценариями, включая параллельные вызовы агентов.